# DSA Week 5 -- Hashing & Indexing

**Course:** Data Structures & Algorithms
**Session:** 3 hours
**Prerequisites:** Weeks 1-4
**Focus:** Hash tables, dict internals, building an Index module

## Learning Objectives

1. Explain how a hash table works (hash function, buckets, collisions)
2. Build a simple hash table from scratch
3. Understand why dict/set lookups are O(1) average
4. Know what causes worst-case O(n) for hash tables (collisions)
5. Build an Index class for fast lookups on any column
6. Benchmark hash-based index vs linear scan

## The Big Idea

A hash table is the most important data structure in practical programming.
Python's `dict` and `set` are both hash tables. They give you O(1) lookup
by converting keys into array indices using a **hash function**.

```
How a hash table works:

  key = "sensor_a"
          |
  hash("sensor_a") = 7429813574  (big number)
          |
  7429813574 % 8 = 6              (index into array of size 8)
          |
  table[6] = ("sensor_a", 25.3)   (store key-value pair)

Lookup: same process in reverse -- O(1)!
```

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Build a Hash Table from Scratch

Let us build a hash table to understand what Python's dict does internally.

```
Hash Table with 5 buckets:

  Bucket 0: [("eve", 81)]
  Bucket 1: [("bob", 92)]
  Bucket 2: [("alice", 88), ("frank", 67)]  <-- collision!
  Bucket 3: []                                <-- empty
  Bucket 4: [("carol", 75)]

Collision: when two keys hash to the same bucket.
Solution: store a list of (key, value) pairs in each bucket.
```

In [ ]:
class SimpleHashTable:
    """A hash table built from scratch for learning."""

    def __init__(self, size=8):
        self.size = size
        self.buckets = [[] for _ in range(size)]
        self.n_items = 0

    def _hash(self, key):
        """Convert key to bucket index."""
        return hash(key) % self.size

    def put(self, key, value):
        """Insert or update key-value pair. O(1) average."""
        idx = self._hash(key)
        bucket = self.buckets[idx]
        # Check if key already exists
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)  # update
                return
        bucket.append((key, value))  # insert
        self.n_items += 1

    def get(self, key, default=None):
        """Look up value by key. O(1) average."""
        idx = self._hash(key)
        for k, v in self.buckets[idx]:
            if k == key:
                return v
        return default

    def __contains__(self, key):
        """Support 'key in table'. O(1) average."""
        idx = self._hash(key)
        for k, v in self.buckets[idx]:
            if k == key:
                return True
        return False

    def show(self):
        """Visualize the hash table."""
        print("  HashTable (" + str(self.n_items) + " items, " + str(self.size) + " buckets):")
        for i, bucket in enumerate(self.buckets):
            if bucket:
                items = ", ".join("(" + repr(k) + ": " + repr(v) + ")" for k, v in bucket)
                print("    Bucket " + str(i) + ": [" + items + "]")
            else:
                print("    Bucket " + str(i) + ": []")

# Demo
ht = SimpleHashTable(5)
for name, score in [("alice", 88), ("bob", 92), ("carol", 75), ("dave", 91), ("eve", 81), ("frank", 67)]:
    ht.put(name, score)
    print("  put(" + repr(name) + ", " + str(score) + ") -> bucket " + str(ht._hash(name)))

print()
ht.show()
print()
print("  get('alice') = " + str(ht.get("alice")))
print("  get('bob')   = " + str(ht.get("bob")))
print("  get('zara')  = " + str(ht.get("zara", "NOT FOUND")))
print("  'carol' in table: " + str("carol" in ht))

---
## Part 2: Understanding Collisions

When two keys hash to the same bucket, it is called a **collision**.
More collisions = slower lookups (worst case: O(n) if everything hashes
to the same bucket).

```
Good hash table (few collisions):
  Bucket 0: [(key1, val1)]
  Bucket 1: [(key2, val2)]
  Bucket 2: [(key3, val3)]
  Bucket 3: [(key4, val4)]
  -> Each lookup: O(1)

Bad hash table (many collisions):
  Bucket 0: [(key1, val1), (key2, val2), (key3, val3), (key4, val4)]
  Bucket 1: []
  Bucket 2: []
  Bucket 3: []
  -> Lookup in bucket 0: O(n) -- degenerates to a list!
```

Python's dict avoids this by:
1. Using a very good hash function
2. Automatically resizing when the table gets too full (load factor > 2/3)

In [ ]:
# Demonstrate the effect of table size on collisions
import random
random.seed(42)

for table_size in [5, 10, 50, 100]:
    ht = SimpleHashTable(table_size)
    for i in range(50):
        ht.put("key_" + str(i), i)

    # Count collisions
    max_bucket = max(len(b) for b in ht.buckets)
    empty = sum(1 for b in ht.buckets if len(b) == 0)
    avg_bucket = ht.n_items / (table_size - empty) if (table_size - empty) > 0 else 0

    print("  Table size " + str(table_size).rjust(3) + ": max bucket = " + str(max_bucket) + ", empty buckets = " + str(empty) + ", avg non-empty = " + "{:.1f}".format(avg_bucket))

print()
print("  Larger table = fewer collisions = faster lookups")
print("  This is why Python dicts auto-resize!")

---
## Part 3: Building a Fast Index for Your Pipeline

An Index lets you look up rows by any column value in O(1) instead of
scanning all rows O(n). This is exactly what databases do internally.

```
Data:
  Row 0: {city: "Cairo", score: 88}
  Row 1: {city: "Alex",  score: 92}
  Row 2: {city: "Cairo", score: 75}
  Row 3: {city: "Luxor", score: 95}

Index on "city":
  "Cairo" -> [0, 2]
  "Alex"  -> [1]
  "Luxor" -> [3]

Query: "Find all Cairo rows"
  Without index: scan all 4 rows -> O(n)
  With index:    index["Cairo"] = [0, 2] -> O(1)!
```

In [ ]:
class Index:
    """A hash-based index for fast lookups on a dataset."""

    def __init__(self, data, key_column):
        """Build index on key_column. O(n) one-time cost."""
        self.key_column = key_column
        self.index = {}

        for i, row in enumerate(data):
            key = row.get(key_column)
            if key not in self.index:
                self.index[key] = []
            self.index[key].append(i)

        print("  Built index on '" + key_column + "': " + str(len(self.index)) + " unique keys from " + str(len(data)) + " rows")

    def lookup(self, key):
        """Find all row indices matching key. O(1)."""
        return self.index.get(key, [])

    def count(self, key):
        """Count rows matching key. O(1)."""
        return len(self.index.get(key, []))

    def unique_keys(self):
        """Return all unique key values."""
        return list(self.index.keys())

# Demo with sample data
data = [
    {"id": 1, "city": "Cairo", "score": 88},
    {"id": 2, "city": "Alex", "score": 92},
    {"id": 3, "city": "Cairo", "score": 75},
    {"id": 4, "city": "Luxor", "score": 95},
    {"id": 5, "city": "Cairo", "score": 60},
    {"id": 6, "city": "Alex", "score": 85},
]

city_idx = Index(data, "city")
print()
print("  Cairo rows: " + str(city_idx.lookup("Cairo")))
print("  Cairo count: " + str(city_idx.count("Cairo")))
print("  Unique cities: " + str(city_idx.unique_keys()))
print()

# Get actual rows
cairo_rows = [data[i] for i in city_idx.lookup("Cairo")]
print("  Cairo data:")
for row in cairo_rows:
    print("    " + str(row))

---
## Part 4: Benchmark -- Linear Scan vs Hash Index

In [ ]:
import timeit
import random

# Generate larger dataset
random.seed(42)
cities = ["Cairo", "Alex", "Luxor", "Giza", "Aswan"]
big_data = [{"city": random.choice(cities), "value": random.random()} for _ in range(100_000)]

# Build index (one-time cost)
idx = Index(big_data, "city")

# Slow: linear scan
def slow_lookup(data, city):
    return [i for i, r in enumerate(data) if r["city"] == city]

# Fast: index lookup
def fast_lookup(idx, city):
    return idx.lookup(city)

print()
# Verify correctness
r_slow = slow_lookup(big_data, "Cairo")
r_fast = fast_lookup(idx, "Cairo")
assert r_slow == r_fast, "Results must match!"
print("  Correctness verified: both return " + str(len(r_slow)) + " rows")
print()

# Benchmark
t_slow = timeit.timeit(lambda: slow_lookup(big_data, "Cairo"), number=100)
t_fast = timeit.timeit(lambda: fast_lookup(idx, "Cairo"), number=100)

print("  Linear scan:  " + "{:.4f}".format(t_slow) + "s (100 queries)")
print("  Index lookup: " + "{:.6f}".format(t_fast) + "s (100 queries)")
print("  Speedup:      " + "{:.0f}".format(t_slow / t_fast) + "x")
print()
print("  The index is built once O(n), then every lookup is O(1).")
print("  Worth it if you do more than ~1 lookup on the same data.")

---
## Common Mistakes

| Mistake | Fix |
|---------|-----|
| Using a list as a dict key | Lists are unhashable. Use a tuple instead |
| Assuming dict preserves insertion order in old Python | True since Python 3.7, but do not rely on it for sorting |
| Not handling missing keys | Use `dict.get(key, default)` instead of `dict[key]` |
| Building index for single-use lookup | Not worth it if you only search once. Linear scan is fine |

---
## Mini-Quiz

In [ ]:
# Q1: What makes a good hash function?
# Answer:

# Q2: What happens when every key hashes to the same bucket?
# Answer:

# Q3: When should you use an Index vs a simple dict vs a list?
# Answer:

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)